# 04 — Detección de anomalías (IsolationForest)

Agregamos eventos por (IP, puerto), entrenamos IsolationForest y marcamos comportamientos inusuales. Es la misma lógica que sirve la API.

In [1]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../datasets/firewall_clean.csv')
agg = df.groupby(['src_ip','dst_port']).size().reset_index(name='eventos')
PUERTOS_RIESGO = {22,23,445,1433,3306,3389,21}
agg['es_riesgo'] = agg['dst_port'].isin(PUERTOS_RIESGO).astype(int)
agg.head()

,src_ip,dst_port,eventos,es_riesgo
0,192.168.0.12,25,13,0
1,192.168.0.12,53,9,0
2,192.168.0.12,80,9,0
3,192.168.0.12,110,10,0
4,192.168.0.12,123,8,0


In [2]:
X = agg[['eventos','dst_port','es_riesgo']].values
Xs = StandardScaler().fit_transform(X)
modelo = IsolationForest(n_estimators=200, contamination=0.08, random_state=42)
agg['anomalia'] = modelo.fit_predict(Xs)
n = (agg['anomalia']==-1).sum()
print(f'Anomalías: {n}/{len(agg)} ({100*n/len(agg):.1f}%)')

Anomalías: 70/876 (8.0%)


## Top combinaciones anómalas

In [3]:
agg[agg['anomalia']==-1].sort_values('eventos', ascending=False).head(10)

,src_ip,dst_port,eventos,es_riesgo,anomalia
855,45.74.13.144,445,186,1,-1
854,45.74.13.144,23,185,1,-1
812,45.21.111.60,21,180,1,-1
874,45.91.57.7,3389,177,1,-1
818,45.21.111.60,3389,175,1,-1
819,45.21.111.60,8080,173,0,-1
836,45.45.125.58,21,169,1,-1
845,45.64.16.8,22,166,1,-1
851,45.64.16.8,8080,166,0,-1
865,45.79.44.152,3306,165,1,-1


## Conclusión
Las anomalías se concentran en **IPs externas (45.x)** atacando puertos de acceso remoto (RDP 3389, SSH 22, SMB 445) con cientos de eventos — patrón típico de fuerza bruta / escaneo. El modelo entrenado se exporta a `api/model.joblib` con `train_model.py` para servirlo vía la API.